# Landsat NDVI → 100m Grid Mapping

## 목적
- Landsat 8/9 Level-2 Surface Reflectance(SR)를 이용해 여름철(6~9월) 평균 NDVI 산출
- 고정된 서울 100m grid(cell_id)에 NDVI 값을 매핑

## 입력 데이터
- GEE Asset: seoul_grid_100m (100m grid)
- Landsat 8/9 Collection 2 T1_L2 (SR_B4, SR_B5)

## 출력 데이터
- cell_id, NDVI (연도별 CSV) → `data_processed/processed/ndvi/`에 저장

In [ ]:
!pip install geemap
!pip install earthengine-api
!pip install python-dotenv

In [4]:
# 기본 라이브러리
import os
import ee
import geemap
from dotenv import load_dotenv

# .env 로드
load_dotenv()

# Earth Engine 인증
# ee.Authenticate()

project_id = "gee-test-1-483406"

# Earth Engine 초기화
ee.Initialize(project=project_id)

In [5]:
# =========================
# Global Configuration
# =========================

# 분석 대상 지역
TARGET_COUNTRY = "Republic of Korea"
TARGET_CITY = "Seoul"

# 기간 설정 (여름철)
START_MMDD = "06-01"
END_MMDD = "09-01"

# Grid 설정
GRID_SIZE = 100  # meters
PROJECTION = "EPSG:3857"

# GEE Asset (100m grid)
GRID_ASSET_ID = f"projects/{project_id}/assets/seoul_grid_100m"

In [6]:
# =========================
# Load Seoul Boundary
# =========================

gaul1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

korea = gaul1.filter(
    ee.Filter.eq("ADM0_NAME", TARGET_COUNTRY)
)

seoul = korea.filter(
    ee.Filter.eq("ADM1_NAME", TARGET_CITY)
)

seoul_geometry = seoul.geometry()

print("Seoul feature count:", seoul.size().getInfo())

Seoul feature count: 1


In [7]:
# =========================
# Load 100m Grid Asset
# =========================

grid_fc = ee.FeatureCollection(GRID_ASSET_ID)

print("Grid cell count:", grid_fc.size().getInfo())

Grid cell count: 92398


In [9]:
# =========================
# Cloud Mask Function (L2 SR)
# =========================

def mask_landsat_l2(img):
    """
    QA_PIXEL 비트 정보를 이용한 구름/눈/그림자 제거
    """
    qa = img.select("QA_PIXEL")

    mask = (
        qa.bitwiseAnd(1 << 1).eq(0)   # Dilated cloud
        .And(qa.bitwiseAnd(1 << 2).eq(0))  # Cirrus
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # Cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # Cloud shadow
        .And(qa.bitwiseAnd(1 << 5).eq(0))  # Snow
    )

    return img.updateMask(mask)

In [10]:
# =========================
# NDVI (Landsat 8/9 SR)
# =========================
# Collection 2 Level-2: SR_B4(RED), SR_B5(NIR)
# Scale factor: SR * 0.0000275 + (-0.2)

def add_ndvi_l8l9_sr(img):
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2)
    nir = img.select("SR_B5").multiply(0.0000275).add(-0.2)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    return img.addBands(ndvi)

In [11]:
# =========================
# Target Year
# =========================

YEAR = 2025

START_DATE = f"{YEAR}-{START_MMDD}"
END_DATE = f"{YEAR}-{END_MMDD}"

print("Analysis period:", START_DATE, "~", END_DATE)

Analysis period: 2025-06-01 ~ 2025-09-01


In [12]:
# =========================
# Landsat 8/9 NDVI Collection
# =========================

l8 = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(seoul_geometry)
    .filter(ee.Filter.lt("CLOUD_COVER", 40))
    .map(mask_landsat_l2)
    .map(add_ndvi_l8l9_sr)
)

l9 = (
    ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(seoul_geometry)
    .filter(ee.Filter.lt("CLOUD_COVER", 40))
    .map(mask_landsat_l2)
    .map(add_ndvi_l8l9_sr)
)

landsat_ndvi = (
    l8.merge(l9)
    .select("NDVI")
)

print("Total NDVI images:", landsat_ndvi.size().getInfo())

Total NDVI images: 5


In [13]:
# =========================
# Summer Mean NDVI Image
# =========================

ndvi_mean_30m = (
    landsat_ndvi
    .mean()
    .clip(seoul_geometry)
    .rename("NDVI")
)

print("Summer mean NDVI image created.")

# NDVI 값 범위 확인 (서울 전체)
ndvi_stats = ndvi_mean_30m.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=seoul_geometry,
    scale=30,
    maxPixels=1e9
)

print("NDVI min/max:", ndvi_stats.getInfo())

Summer mean NDVI image created.
NDVI min/max: {'NDVI_max': 30.322541259066064, 'NDVI_min': -23.50453574569682}


In [14]:
# =========================
# Map NDVI to 100m Grid
# =========================

grid_with_ndvi = ndvi_mean_30m.reduceRegions(
    collection=grid_fc,
    reducer=ee.Reducer.mean(),
    scale=GRID_SIZE  # 100m
)

print("NDVI mapping to grid completed.")

NDVI mapping to grid completed.


In [15]:
# =========================
# Clean output columns
# =========================

def rename_mean_to_ndvi(feat):
    return (
        feat
        .set("NDVI", feat.getNumber("mean"))
        .select(["cell_id", "NDVI"])
    )

grid_with_ndvi_final = grid_with_ndvi.map(rename_mean_to_ndvi)

In [16]:
# =========================
# Coverage check
# =========================

total_cells = grid_fc.size()
ndvi_cells = grid_with_ndvi_final.size()

print("Total grid cells:", total_cells.getInfo())
print("Cells with NDVI:", ndvi_cells.getInfo())
print(
    "Coverage (%):",
    (ndvi_cells.divide(total_cells).multiply(100)).getInfo()
)

Total grid cells: 92398
Cells with NDVI: 92398
Coverage (%): 100


In [19]:
# =========================
# 시각화: 다운로드 전 데이터 확인
# =========================
# 그리드 NDVI를 이미지로 변환해 지도에 표시

ndvi_grid_image = grid_with_ndvi_final.reduceToImage(
    properties=["NDVI"],
    reducer=ee.Reducer.first()
)

Map = geemap.Map()
Map.centerObject(seoul, 11)

Map.addLayer(
    ndvi_grid_image,
    {
        "min": -0.2,
        "max": 0.9,
        "palette": ["brown", "yellow", "lightgreen", "green", "darkgreen"],
    },
    "NDVI 100m Grid (여름 평균)",
)

Map.addLayer(seoul.style(**{"color": "gray", "fillColor": "00000000"}), {}, "Seoul 경계")

Map

Map(center=[37.53803118323625, 127.00584943229947], controls=(WidgetControl(options=['position', 'transparent_…

In [18]:
# =========================
# Export to CSV
# =========================

import pandas as pd

output_dir = "../data_processed/processed/ndvi"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(
    output_dir,
    f"ndvi_{YEAR}_summer_100m.csv"
)

export_ndvi_csv = grid_with_ndvi_final.select([
    "cell_id",
    "NDVI"
])

print(f"Exporting NDVI CSV to {csv_path}...")

geemap.ee_export_vector(
    export_ndvi_csv,
    filename=csv_path
)

# GEE export에 포함된 system:index 등 인덱스 컬럼 제거 후 인덱스 없이 저장
df = pd.read_csv(csv_path)
cols_to_drop = [c for c in df.columns if c in ("system:index", "system_index")]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
df.to_csv(csv_path, index=False)

print("NDVI CSV export complete.")

Exporting NDVI CSV to ../data_processed/processed/ndvi\ndvi_2025_summer_100m.csv...
Generating URL ...
Please wait ...
Data downloaded to c:\Users\User\Downloads\data_ai\data_ai\data_processed\processed\ndvi\ndvi_2025_summer_100m.csv
NDVI CSV export complete.
